In [1]:
from datasets import load_dataset
import json
import pandas as pd
import os
from PIL import Image
from tqdm import tqdm

ModuleNotFoundError: No module named 'datasets'

# 处理数据集

In [5]:
ds = load_dataset("crag-mm-2025/crag-mm-single-turn-public", "validation")
ds

Dataset({
    features: ['id', 'modality', 'text', 'image'],
    num_rows: 3743
})

In [6]:
img_dir = "images"

os.makedirs(img_dir, exist_ok=True)

In [ ]:
import re
df = pd.read_csv('data/datasets/evqa/test.csv', nrows=3751)
answer_pairs = {}
for t, (q, a) in zip(df['wikipedia_title'], zip(df['question'], df['answer'])):
    delimiter = r"\|"
    result_split = re.split(delimiter, a)
    results = []
    for result in result_split:
        results.append(result.strip())
    key = f"{t}:{q}"
    answer_pairs[key] = results

processed_data = []
for item in ds:
    image = item['image']
    img_filename = f"{item['id']}.jpg"
    img_filepath = os.path.join(img_dir, img_filename)
    
    # if image.mode != 'RGB':
    #     image = image.convert('RGB')

    # image.save(img_filepath, 'JPEG')
    answer = answer_pairs[item['text']]
    clean_item = {
        "id": item["id"],
        "question": item["text"],
        "golden_answers": answer
    }
    processed_data.append(clean_item)


In [22]:
output_filename = "data/datasets/evqa/test.jsonl"

with open(output_filename, 'w', encoding='utf-8') as f:
    for entry in tqdm(processed_data, desc="Writing jsonl"):
        if not isinstance(entry, dict):
            try:
                entry = dict(entry)
            except Exception:
                continue
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"Saved {len(processed_data)} entries to {output_filename}")

Writing jsonl: 100%|██████████| 3743/3743 [00:00<00:00, 257711.68it/s]

Saved 3743 entries to data/datasets/evqa/test.jsonl


# 处理语料库为jsonl

In [3]:
ds = load_dataset("izhx/UMRB-EncyclopediaVQA", "corpus", split="corpus")
ds

Dataset({
    features: ['id', 'modality', 'text', 'image'],
    num_rows: 68313
})

In [7]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True 
BATCH_SIZE=10000

corpus_items = []
img_dir = 'data/datasets/evqa/corpus/images'
with open('data/datasets/evqa/corpus/corpus.jsonl', 'w', encoding='utf-8') as f:
    for item in tqdm(ds, desc="Begin constructing corpus.jsonl"):
            id = item['id']
            text = item['text']
            image = item['image']

            corpus_item = {
                "id": id,
                "text": text,
                "image": image
            }
            corpus_items.append(corpus_item)
            if len(corpus_items) >= BATCH_SIZE:
                for item in corpus_items:
                    # image = item['image']
                    img_filename = f"{item['id']}.jpg"
                    img_filepath = os.path.join(img_dir, img_filename)
                    
                    # if image.mode != 'RGB':
                    #     image = image.convert('RGB')

                    # image.save(img_filepath, 'JPEG')
                    item_wo_image = {
                            "id": item['id'],
                            "contents": item['text'],
                            "image": img_filepath
                    }
                    f.write(json.dumps(item_wo_image, ensure_ascii=False) + "\n")
                corpus_items = []
    if corpus_items:
        for item in corpus_items:
            # image = item['image']
            img_filename = f"{item['id']}.jpg"
            img_filepath = os.path.join(img_dir, img_filename)
            
            # if image.mode != 'RGB':
            #     image = image.convert('RGB')

            # image.save(img_filepath, 'JPEG')
            item_wo_image = {
                    "id": item['id'],
                    "contents": item['text'],
                    "image": img_filepath
            }
            f.write(json.dumps(item_wo_image, ensure_ascii=False) + "\n")
        corpus_items = []
                           

Begin constructing corpus.jsonl: 100%|██████████| 68313/68313 [06:43<00:00, 169.16it/s]


# 除去CRAG中的多跳类问题

In [1]:
from datasets import load_dataset
import json
import pandas as pd
import os
from PIL import Image
from tqdm import tqdm
ds = load_dataset("crag-mm-2025/crag-mm-single-turn-public", "validation")
ds

/root/miniconda3/envs/rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: BuilderConfig 'validation' not found. Available: ['default']